# Notebook: Introduction to Pydantic

[Pydantic](https://docs.pydantic.dev/) is a Python library for defining the *shape* of your data as a class, and then validating and parsing data against it automatically.

It's the library that powers `response_format=...` in the structured outputs notebook — this notebook covers the fundamentals of Pydantic itself, independent of LLMs, one small step at a time.

## 1. A Basic Model

A Pydantic model is a class that inherits from `BaseModel` — but it looks different from the Python classes you've learned so far. A regular Python class defines its attributes inside `__init__`, like this:

```python
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age
```

A Pydantic model skips `__init__` entirely. Instead, you declare each attribute directly in the class body, as a **type-annotated class attribute** — just a name followed by `: type`, with no value assigned:

```python
class Person(BaseModel):
    name: str
    age: int
```

Pydantic reads these annotations and automatically generates the constructor, validation, and attribute access for you. You still create and use instances the normal way (`Person(name="Alice", age=30)`, `person.name`) — you just never write `__init__` yourself.

In [1]:
from pydantic import BaseModel

class Person(BaseModel):
    name: str
    age: int

person = Person(name="Alice", age=30)
print(person)
print("Name:", person.name)
print("Age:", person.age)

name='Alice' age=30
Name: Alice
Age: 30


Pydantic also converts compatible types automatically — this is called **coercion**. Here, the string `"25"` is automatically converted to the integer `25`, since that's what the `age: int` annotation expects.

In [2]:
person2 = Person(name="Bob", age="25")
print(person2)
print("Type of age:", type(person2.age))

name='Bob' age=25
Type of age: <class 'int'>


## 2. Validation Errors

If the data doesn't match the schema, Pydantic raises a `ValidationError` instead of silently accepting bad data — this is the whole point of using it. This is especially useful when parsing data from an external source you don't fully control, like an LLM's output.

In [3]:
from pydantic import ValidationError

try:
    Person(name="Charlie", age="not a number")
except ValidationError as e:
    print("Validation failed as expected:")
    print(e)

Validation failed as expected:
1 validation error for Person
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='not a number', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing


## 3. `Field()` for Extra Constraints

`Field()` lets you add validation rules beyond just the type, such as numeric ranges (`ge`/`le`/`gt`/`lt`, meaning "greater/less than (or equal)") or string length (`min_length`/`max_length`).

In [4]:
from pydantic import Field

class Product(BaseModel):
    name: str = Field(min_length=1)
    price: float = Field(gt=0)            # must be greater than 0
    quantity: int = Field(ge=0, le=1000)  # between 0 and 1000 (inclusive)

valid_product = Product(name="Coffee Mug", price=9.99, quantity=50)
print(valid_product)

name='Coffee Mug' price=9.99 quantity=50


In [5]:
try:
    Product(name="Broken Item", price=-5.0, quantity=50)
except ValidationError as e:
    print("Validation failed as expected (negative price):")
    print(e)

Validation failed as expected (negative price):
1 validation error for Product
price
  Input should be greater than 0 [type=greater_than, input_value=-5.0, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than


## 4. Restricting Values with `Literal`

`Literal` restricts a field to a fixed set of allowed values — useful for categories, statuses, or anything an LLM should classify into fixed options (we used exactly this for sentiment classification in the structured outputs notebook).

In [6]:
from typing import Literal

class Task(BaseModel):
    title: str
    status: Literal["todo", "in_progress", "done"]

task = Task(title="Write report", status="in_progress")
print(task)

title='Write report' status='in_progress'


In [7]:
try:
    Task(title="Write report", status="urgent")  # not one of the allowed values
except ValidationError as e:
    print("Validation failed as expected (invalid status):")
    print(e)

Validation failed as expected (invalid status):
1 validation error for Task
status
  Input should be 'todo', 'in_progress' or 'done' [type=literal_error, input_value='urgent', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error


## 5. Optional Fields and Default Values

`Optional[X]` means the field can be `X` or `None`. Giving it `= None` (or another default value) makes the field optional to provide when creating the object — without a default, Pydantic requires every field to be passed in.

In [8]:
from typing import Optional

class Contact(BaseModel):
    name: str
    email: Optional[str] = None
    newsletter_signup: bool = False  # a plain default, not Optional

contact_minimal = Contact(name="Dana")
print(contact_minimal)

contact_full = Contact(name="Eli", email="eli@example.com", newsletter_signup=True)
print(contact_full)

name='Dana' email=None newsletter_signup=False
name='Eli' email='eli@example.com' newsletter_signup=True


## 6. Nested Models

Models can contain other models as fields — Pydantic validates the whole structure recursively, including lists of nested models. This is exactly the pattern we used for receipts and purchase orders in the structured outputs notebook.

In [9]:
class Address(BaseModel):
    city: str
    zip_code: str

class Customer(BaseModel):
    name: str
    address: Address                        # a single nested model
    previous_addresses: list[Address] = []  # a list of nested models

customer = Customer(
    name="Frank",
    address={"city": "Munich", "zip_code": "80331"},  # a dict is automatically parsed into an Address
    previous_addresses=[{"city": "Berlin", "zip_code": "10115"}],
)
print(customer)
print("City:", customer.address.city)

name='Frank' address=Address(city='Munich', zip_code='80331') previous_addresses=[Address(city='Berlin', zip_code='10115')]
City: Munich


## 7. Custom Validators

For rules that go beyond a type or `Field()` (e.g. checking a specific format), use `@field_validator` to write your own validation function for a field.

In [10]:
from pydantic import field_validator

class User(BaseModel):
    username: str

    @field_validator("username")
    @classmethod
    def username_must_be_lowercase(cls, value):
        if value != value.lower():
            raise ValueError("username must be lowercase")
        return value

print(User(username="nils"))

username='nils'


In [11]:
try:
    User(username="Nils")
except ValidationError as e:
    print("Validation failed as expected (not lowercase):")
    print(e)

Validation failed as expected (not lowercase):
1 validation error for User
username
  Value error, username must be lowercase [type=value_error, input_value='Nils', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


## 8. Converting to/from JSON and Dicts

These four methods cover the common conversions between a Pydantic model and plain Python/JSON data. `model_validate_json()` in particular is exactly how you'd parse a raw JSON string returned by an LLM into a validated object.

In [12]:
person = Person(name="Grace", age=28)

# model -> dict / JSON string
print("model_dump():     ", person.model_dump())
print("model_dump_json():", person.model_dump_json())

model_dump():      {'name': 'Grace', 'age': 28}
model_dump_json(): {"name":"Grace","age":28}


In [13]:
# dict / JSON string -> model
print("model_validate():     ", Person.model_validate({"name": "Grace", "age": 28}))
print("model_validate_json():", Person.model_validate_json('{"name": "Grace", "age": 28}'))

model_validate():      name='Grace' age=28
model_validate_json(): name='Grace' age=28


## 9. Generating a JSON Schema

`model_json_schema()` turns a Pydantic model into a JSON Schema dictionary. This is exactly what happens under the hood when you pass a Pydantic model as `response_format` to an LLM in the structured outputs notebook: the schema is sent to the model (or the inference server) to constrain its output to match this exact structure.

In [14]:
import json

print(json.dumps(Product.model_json_schema(), indent=2))

{
  "properties": {
    "name": {
      "minLength": 1,
      "title": "Name",
      "type": "string"
    },
    "price": {
      "exclusiveMinimum": 0,
      "title": "Price",
      "type": "number"
    },
    "quantity": {
      "maximum": 1000,
      "minimum": 0,
      "title": "Quantity",
      "type": "integer"
    }
  },
  "required": [
    "name",
    "price",
    "quantity"
  ],
  "title": "Product",
  "type": "object"
}
